In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline


def init_spark_session(app_name: str = "CommercialLoanRiskTraining") -> SparkSession:
    """Initializes a Spark session configured for local Windows execution."""
    # Stop any existing stale session
    SparkSession.builder.getOrCreate().stop()
    
    return SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.host", "localhost") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()


def preprocess_data(df):
    """Fills missing values, calculates financial ratios, and encodes target column."""
    fill_defaults = {
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": "0",
        "Self_Employed": "No",
        "LoanAmount": 146.4,
        "Loan_Amount_Term": 360,
        "Credit_History": 1.0
    }
    
    return df.na.fill(fill_defaults) \
        .withColumn("Total_Income", col("ApplicantIncome") + col("CoapplicantIncome")) \
        .withColumn(
            "Debt_To_Income",
            when(col("Total_Income") > 0, col("LoanAmount") / col("Total_Income")).otherwise(0.0)
        ) \
        .withColumn("label", when(col("Loan_Status") == "Y", 1.0).otherwise(0.0))


def build_pipeline():
    """Constructs the PySpark ML feature transformation and modeling pipeline."""
    cat_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"]
    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
    encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") for c in cat_cols]

    num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term", "Credit_History", "Total_Income", "Debt_To_Income"]
    assembler_inputs = [f"{c}_vec" for c in cat_cols] + num_cols
    assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

    classifier = DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=5,
        seed=42
    )

    return Pipeline(stages=indexers + encoders + [assembler, classifier])


def main():
    spark = init_spark_session()

    data_path = r"C:\Users\Lincoln\Downloads\Data\1_Training_Data.csv"
    model_output_path = r"C:\Users\Lincoln\Downloads\Data\production_loan_model"

    # Data Ingestion
    raw_df = spark.read.csv(data_path, header=True, inferSchema=True)
    df_cleaned = preprocess_data(raw_df)

    # Model Training
    train_df, test_df = df_cleaned.randomSplit([0.8, 0.2], seed=42)
    pipeline = build_pipeline()
    pipeline_model = pipeline.fit(train_df)

    # Evaluation
    predictions = pipeline_model.transform(test_df)
    evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    roc_auc = evaluator.evaluate(predictions)
    print(f"Model Training Complete. Test Area Under ROC: {roc_auc:.4f}")

    # Artifact Serialization
    pipeline_model.write().overwrite().save(model_output_path)
    print(f"PipelineModel successfully saved to {model_output_path}")


if __name__ == "__main__":
    main()

Model Training Complete. Test Area Under ROC: 0.7360
PipelineModel successfully saved to C:\Users\Lincoln\Downloads\Data\production_loan_model
